## Ran on Colab

In [1]:
from pathlib import Path
import numpy as np
import librosa as lb
from sklearn.model_selection import KFold
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!unzip "/content/drive/MyDrive/UVIC Y5/475/giantsteps-tempo-dataset.zip" -d /content/

Archive:  /content/drive/MyDrive/UVIC Y5/475/giantsteps-tempo-dataset.zip
replace /content/giantsteps-tempo-dataset/.DS_Store? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [4]:
audio_path = Path("/content/giantsteps-tempo-dataset/audio")
label_path = Path("/content/giantsteps-tempo-dataset/annotations/tempo")

# Load tempo labels
tempo_labels = {}
for file in label_path.iterdir():
    tempo_labels[file.stem] = float(file.read_text().strip())

# Load dataset
dataset = []
for file in audio_path.iterdir():
    track_id = file.stem
    if track_id in tempo_labels:
        dataset.append((file, tempo_labels[track_id]))

print(f"Loaded {len(dataset)} audio tracks with tempo labels")


Loaded 664 audio tracks with tempo labels


In [5]:
def precompute_tempograms(dataset, sr=22050, hop_length=512, n_bins=96, duration=30):
    """Compute tempogram for each track and return as fixed-size array."""
    all_tg = []
    all_labels = []

    for path, bpm in tqdm(dataset, desc="Precomputing tempograms"):
        y_audio, _ = lb.load(path, duration=duration, sr=sr)

        onset_env = lb.onset.onset_strength(y=y_audio, sr=sr, hop_length=hop_length)
        tg = lb.feature.tempogram(onset_envelope=onset_env, sr=sr, hop_length=hop_length)

        # Trim to n_bins BPM bins (tempogram rows = BPM frequencies)
        tg = tg[:n_bins, :]

        # Normalize
        if tg.max() > 0:
            tg = tg / tg.max()

        # Fixed number of time frames
        n_frames = int(duration * sr / hop_length)
        if tg.shape[1] < n_frames:
            tg = np.pad(tg, ((0, 0), (0, n_frames - tg.shape[1])))
        else:
            tg = tg[:, :n_frames]

        all_tg.append(tg.astype(np.float32))
        all_labels.append(bpm)

    return np.array(all_tg), np.array(all_labels, dtype=np.float32)


tg_data, labels = precompute_tempograms(dataset)

Precomputing tempograms:   0%|          | 0/664 [00:00<?, ?it/s]

In [6]:
class TempogramDataset(Dataset):
    def __init__(self, tg_data, labels):
        self.tg_data = tg_data
        self.labels  = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Shape: (1, n_bins, n_frames)
        tg  = torch.tensor(self.tg_data[idx], dtype=torch.float32).unsqueeze(0)
        bpm = torch.tensor(self.labels[idx],  dtype=torch.float32)
        return tg, bpm

In [ ]:
class TempoCNN(nn.Module):
    def __init__(self, n_bins=96):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1 — wide time kernel to capture beat periodicity
            nn.Conv2d(1, 32, kernel_size=(3, 15), padding=(1, 7)),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=(3, 15), padding=(1, 7), stride=(1, 2)),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            # Block 2 — compress frequency axis, keep time
            nn.Conv2d(32, 64, kernel_size=(5, 7), padding=(2, 3)),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=(5, 7), padding=(2, 3), stride=(2, 2)),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            # Block 3 — deeper features
            nn.Conv2d(64, 128, kernel_size=(3, 7), padding=(1, 3)),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=(3, 7), padding=(1, 3), stride=(2, 2)),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # Global average pool — collapses spatial dims to 1x1
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.regressor(x)
        return x.squeeze(1)

In [8]:
def half_double_corrected_mae(y_true, y_pred):
    candidates = np.vstack([y_pred, y_pred*2, y_pred/2])
    errors = np.abs(candidates - y_true)
    return np.mean(np.min(errors, axis=0))


def acc1(y_true, y_pred, tol=0.04):
    return np.mean(np.abs(y_pred - y_true) <= tol * y_true)


def acc2(y_true, y_pred, tol=0.04):
    candidates = np.vstack([y_pred, y_pred*2, y_pred/2])
    errors = np.abs(candidates - y_true)
    min_errors = np.min(errors, axis=0)
    return np.mean(min_errors <= tol * y_true)

In [9]:
def cnn_training_loop(tg_data, labels, n_epochs=80, batch_size=32, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    indices = np.arange(len(tg_data))

    all_mae = []
    all_acc1 = []
    all_acc2 = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(indices)):
        print(f"\nFold {fold+1}")

        train_ds = TempogramDataset(tg_data[train_idx], labels[train_idx])
        test_ds  = TempogramDataset(tg_data[test_idx],  labels[test_idx])

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0)
        test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=0)

        model     = TempoCNN().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        # Gentler schedule — halve lr every 25 epochs instead of 10
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=25, gamma=0.5)
        criterion = nn.L1Loss()

        # ── Train ──
        for epoch in tqdm(range(n_epochs), desc=f"Fold {fold+1} epochs"):
            model.train()
            epoch_loss = 0
            for tg, bpm in train_loader:
                tg, bpm = tg.to(device), bpm.to(device)
                optimizer.zero_grad()
                pred = model(tg)
                loss = criterion(pred, bpm)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            scheduler.step()

            if (epoch + 1) % 20 == 0:
                print(f"  Epoch {epoch+1}/{n_epochs}  Loss: {epoch_loss/len(train_loader):.3f}")

        # ── Evaluate ──
        model.eval()
        all_preds, all_targets = [], []

        with torch.no_grad():
            for tg, bpm in test_loader:
                tg = tg.to(device)
                pred = model(tg)
                all_preds.extend(pred.cpu().numpy())
                all_targets.extend(bpm.numpy())

        y_pred = np.array(all_preds)
        y_test = np.array(all_targets)

        errors = np.abs(y_pred - y_test)
        print(f"Errors > 30 BPM: {np.sum(errors > 30)} / {len(errors)}")
        print(f"Errors > 60 BPM: {np.sum(errors > 60)} / {len(errors)}")

        mae        = half_double_corrected_mae(y_test, y_pred)
        acc1_score = acc1(y_test, y_pred)
        acc2_score = acc2(y_test, y_pred)

        print("\nFold Results")
        print(f"MAE (half/double corrected): {mae:.2f} BPM")
        print(f"Acc1 (±4% strict): {acc1_score:.3f}")
        print(f"Acc2 (±4% half/double): {acc2_score:.3f}")

        all_mae.append(mae)
        all_acc1.append(acc1_score)
        all_acc2.append(acc2_score)

    print("\nResults")
    print(f"MAE (half/double corrected): {np.mean(all_mae):.2f} ± {np.std(all_mae):.2f} BPM")
    print(f"Acc1 (±4% strict): {np.mean(all_acc1):.3f} ± {np.std(all_acc1):.3f}")
    print(f"Acc2 (±4% half/double): {np.mean(all_acc2):.3f} ± {np.std(all_acc2):.3f}")


cnn_training_loop(tg_data, labels)

Using device: cuda

Fold 1


Fold 1 epochs:   0%|          | 0/80 [00:00<?, ?it/s]

  Epoch 20/80  Loss: 21.720
  Epoch 40/80  Loss: 20.229
  Epoch 60/80  Loss: 19.646
  Epoch 80/80  Loss: 18.993
Errors > 30 BPM: 20 / 133
Errors > 60 BPM: 6 / 133

Fold Results
MAE (half/double corrected): 10.01 BPM
Acc1 (±4% strict): 0.368
Acc2 (±4% half/double): 0.383

Fold 2


Fold 2 epochs:   0%|          | 0/80 [00:00<?, ?it/s]

  Epoch 20/80  Loss: 22.727
  Epoch 40/80  Loss: 21.533
  Epoch 60/80  Loss: 20.959
  Epoch 80/80  Loss: 20.379
Errors > 30 BPM: 24 / 133
Errors > 60 BPM: 10 / 133

Fold Results
MAE (half/double corrected): 10.07 BPM
Acc1 (±4% strict): 0.429
Acc2 (±4% half/double): 0.451

Fold 3


...

  Epoch 20/80  Loss: 21.013
  Epoch 40/80  Loss: 19.678
  Epoch 60/80  Loss: 19.472
  Epoch 80/80  Loss: 18.686
Errors > 30 BPM: 26 / 133
Errors > 60 BPM: 10 / 133

Fold Results
MAE (half/double corrected): 10.82 BPM
Acc1 (±4% strict): 0.316
Acc2 (±4% half/double): 0.376

Fold 4


Fold 4 epochs:   0%|          | 0/80 [00:00<?, ?it/s]

  Epoch 20/80  Loss: 23.491
  Epoch 40/80  Loss: 20.678
  Epoch 60/80  Loss: 20.486
  Epoch 80/80  Loss: 19.818
Errors > 30 BPM: 19 / 133
Errors > 60 BPM: 8 / 133

Fold Results
MAE (half/double corrected): 8.55 BPM
Acc1 (±4% strict): 0.511
Acc2 (±4% half/double): 0.526

Fold 5


Fold 5 epochs:   0%|          | 0/80 [00:00<?, ?it/s]

  Epoch 20/80  Loss: 23.189
  Epoch 40/80  Loss: 20.753
  Epoch 60/80  Loss: 21.443
  Epoch 80/80  Loss: 20.868
Errors > 30 BPM: 22 / 132
Errors > 60 BPM: 8 / 132

Fold Results
MAE (half/double corrected): 10.10 BPM
Acc1 (±4% strict): 0.417
Acc2 (±4% half/double): 0.424

Results
MAE (half/double corrected): 9.91 ± 0.74 BPM
Acc1 (±4% strict): 0.408 ± 0.065
Acc2 (±4% half/double): 0.432 ± 0.054
